# 11.17 — TRPO & PPO

TRPO and PPO are policy-gradient methods for the same practical problem: an RL agent can improve only by changing its action probabilities, but a probability jump that is too large can destroy the data distribution it just learned from. In this lesson, we build the policy-gradient objective, the trust-region KL constraint, and PPO's clipped surrogate from scratch with NumPy, then train a tiny toy policy without any RL library.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build TRPO and PPO one idea at a time. Run each cell in order and read the printed intermediate values — the policy probabilities, probability ratios, advantages, KL distances, and clipped objectives are all shown directly. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, softmax probabilities, and small optimization loops.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for rollouts and policy updates.

### 1. Policies, returns, and advantages

A policy is a probability distribution over actions. In the smallest two-action example, logits become probabilities with softmax, expected reward is a probability-weighted average, and the **advantage** says whether the sampled action did better than the policy's own baseline. Policy gradients do not push every rewarded action equally; they push actions with positive advantage up and actions with negative advantage down.

In [ ]:
logits_w = np.array([1.0, 0.0])                         # current preferences for actions left/right.
exp_w = np.exp(logits_w - np.max(logits_w))              # stable exponentials for softmax.
pi_w = exp_w / exp_w.sum()                               # action probabilities.
rewards_w = np.array([2.0, 0.0])                         # one-step reward for each action in this toy state.
expected_w = float(pi_w @ rewards_w)                     # V(s) under the current policy.
advantages_w = rewards_w - expected_w                    # A(s,a)=Q(s,a)-V(s).
print("policy probabilities:", np.round(pi_w, 3))
print("expected reward V:", round(expected_w, 3))
print("advantages:", np.round(advantages_w, 3))
assert np.allclose(np.round(pi_w, 3), [0.731, 0.269])
assert round(expected_w, 3) == 1.462

▶ What you'll see: action 0 has probability 0.731 and positive advantage, while action 1 has negative advantage.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["a0", "a1"], advantages_w, color=["seagreen", "indianred"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("1: advantage = action value minus baseline")
plt.ylabel("A(s,a)")
plt.show()

▶ What you'll see: one bar above zero and one below zero — the policy-gradient signal is directional, not just rewarding high raw returns.

*Why it's done this way:* The gradient identity $\nabla_\theta J=\mathbb{E}[\nabla_\theta\log\pi_\theta(a\mid s)A_t]$ says to change log-probability in proportion to advantage. Subtracting the baseline $V(s)$ lowers variance because common reward shared by all actions cancels out; only the action's relative usefulness remains.

### 2. Importance ratios: reusing old-policy data

TRPO and PPO collect trajectories with an **old** policy, then ask whether a **new** policy would make the sampled actions more or less likely. The ratio $r_t(\theta)=\pi_\theta(a_t\mid s_t)/\pi_{old}(a_t\mid s_t)$ is the bridge: values above 1 mean the new policy emphasizes that sampled action; values below 1 mean it deemphasizes it.

In [ ]:
old_pi_w = np.array([0.60, 0.40])                         # behavior policy that collected data.
new_pi_w = np.array([0.78, 0.22])                         # candidate updated policy.
actions_w = np.array([0, 0, 1, 0, 1])                     # actions observed in a tiny batch.
adv_w = np.array([1.2, 0.7, -0.5, 0.4, -1.0])             # estimated advantages for those samples.
ratios_w = new_pi_w[actions_w] / old_pi_w[actions_w]      # likelihood ratio per sampled action.
unclipped_terms_w = ratios_w * adv_w                      # vanilla surrogate terms.
print("ratios:", np.round(ratios_w, 3))
print("ratio * advantage:", np.round(unclipped_terms_w, 3))
assert np.allclose(np.round(ratios_w, 3), [1.3, 1.3, 0.55, 1.3, 0.55])

▶ What you'll see: action 0 samples are amplified by 1.3, while action 1 samples are discounted to 0.55.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.scatter(ratios_w, adv_w, s=90, color="teal")
plt.axvline(1.0, color="black", linewidth=0.8)
plt.axhline(0.0, color="black", linewidth=0.8)
plt.xlabel("probability ratio r")
plt.ylabel("advantage A")
plt.title("2: old data weighted by new/old probability")
plt.show()

▶ What you'll see: positive-advantage points want ratios above 1, while negative-advantage points are helped by ratios below 1.

*Why it's done this way:* We cannot pretend old-policy samples came from the new policy for free. The ratio is an importance weight correcting that mismatch: if the new policy makes a sampled action 30% more likely, that sample counts 30% more in the surrogate objective. The danger is that extreme ratios produce high-variance, destructive updates.

### 3. TRPO's trust region: constrain the KL jump

TRPO keeps the policy update inside a **trust region**: maximize the surrogate objective, but only if the average KL divergence from old policy to new policy stays below a small budget $\delta$. KL is not a reward; it is a distance-like measure of how much the action distribution changed. Small KL means the new policy still lives near the data-generating policy, so the surrogate remains trustworthy.

In [ ]:
def kl_w(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(np.sum(p * (np.log(p) - np.log(q))))

candidate_ps_w = np.linspace(0.50, 0.95, 10)              # candidate probability for action 0.
old_dist_w = np.array([0.60, 0.40])
delta_w = 0.02
kls_w = np.array([kl_w(old_dist_w, [p, 1 - p]) for p in candidate_ps_w])
allowed_w = kls_w <= delta_w
print("candidate p(a0):", np.round(candidate_ps_w, 2))
print("KL(old||new):", np.round(kls_w, 4))
print("allowed by δ=0.02:", allowed_w.astype(int))
assert allowed_w.sum() == 3

▶ What you'll see: only candidates near the old 0.60/0.40 distribution fit under the KL budget.

In [ ]:
surrogate_w = candidate_ps_w * 2.0 + (1 - candidate_ps_w) * 0.0  # expected one-step reward in the toy state.
best_free_w = candidate_ps_w[np.argmax(surrogate_w)]
best_trpo_w = candidate_ps_w[allowed_w][np.argmax(surrogate_w[allowed_w])]
print("best unconstrained p(a0):", round(float(best_free_w), 2))
print("best trust-region p(a0):", round(float(best_trpo_w), 2))
plt.figure(figsize=(5, 3))
plt.plot(candidate_ps_w, surrogate_w, marker="o", label="surrogate")
plt.fill_between(candidate_ps_w, surrogate_w.min(), surrogate_w.max(), where=allowed_w, alpha=0.18, color="seagreen", label="KL ≤ δ")
plt.axvline(best_trpo_w, color="seagreen", linestyle="--", label="TRPO choice")
plt.title("3: trust region limits the greedy jump")
plt.xlabel("candidate probability of action 0")
plt.ylabel("surrogate value")
plt.legend()
plt.show()

▶ What you'll see: the reward surrogate prefers the largest probability, but the trust region stops at the best candidate that remains close enough.

*Why it's done this way:* A policy-gradient estimate is local: it tells us which direction looked good under the old policy's state-action distribution. KL constrains the step so the new policy does not move into a region where that estimate is no longer valid. TRPO solves this as a constrained optimization problem rather than choosing a plain learning rate.

### 4. PPO's clipped surrogate: a cheap trust-region proxy

PPO replaces TRPO's constrained solve with a simple clipped objective. For positive advantage, increasing the ratio helps only until $1+\epsilon$; beyond that the objective is flat. For negative advantage, decreasing the ratio helps only until $1-\epsilon$. The `min` in PPO chooses the more conservative of the unclipped and clipped terms.

In [ ]:
eps_w = 0.2
ratio_grid_w = np.linspace(0.4, 1.6, 61)
pos_adv_w = 1.0
neg_adv_w = -1.0
clip_grid_w = np.clip(ratio_grid_w, 1 - eps_w, 1 + eps_w)
pos_obj_w = np.minimum(ratio_grid_w * pos_adv_w, clip_grid_w * pos_adv_w)
neg_obj_w = np.minimum(ratio_grid_w * neg_adv_w, clip_grid_w * neg_adv_w)
print("clip interval:", (1 - eps_w, 1 + eps_w))
idx15_w = int(np.argmin(np.abs(ratio_grid_w - 1.5)))
idx05_w = int(np.argmin(np.abs(ratio_grid_w - 0.5)))
print("positive A at r=1.5:", round(float(pos_obj_w[idx15_w]), 3))
print("negative A at r=0.5:", round(float(neg_obj_w[idx05_w]), 3))
assert round(float(pos_obj_w[idx15_w]), 3) == 1.2
assert round(float(neg_obj_w[idx05_w]), 3) == -0.8

▶ What you'll see: ratios outside [0.8, 1.2] stop improving the objective in the direction that would make the update too aggressive.

In [ ]:
plt.figure(figsize=(5.2, 3.2))
plt.plot(ratio_grid_w, ratio_grid_w * pos_adv_w, color="gray", linestyle="--", label="unclipped A=+1")
plt.plot(ratio_grid_w, pos_obj_w, color="seagreen", label="PPO clipped A=+1")
plt.plot(ratio_grid_w, ratio_grid_w * neg_adv_w, color="lightgray", linestyle="--", label="unclipped A=-1")
plt.plot(ratio_grid_w, neg_obj_w, color="indianred", label="PPO clipped A=-1")
plt.axvspan(1 - eps_w, 1 + eps_w, alpha=0.12, color="steelblue", label="no-clip band")
plt.xlabel("ratio r = π_new / π_old")
plt.ylabel("surrogate term")
plt.title("4: PPO clipping removes incentive for huge ratios")
plt.legend(fontsize=8)
plt.show()

▶ What you'll see: the green curve flattens above 1.2 and the red curve flattens below 0.8 — that flatness is the safety mechanism.

*Why it's done this way:* PPO does not forbid large KL directly. Instead, it removes the objective's reward for probability ratios that move too far in the advantage-improving direction. This is cheaper than TRPO's constrained second-order update, yet it preserves the same local-update intuition: improve, but do not let a single batch justify a huge probability jump.

### 5. Training a toy PPO policy from scratch

Now we put the pieces together in a one-state, two-action environment. Action 0 gives reward 1 and action 1 gives reward 0, so the optimal policy should increase action 0. We sample from the old policy, compute advantages from the batch mean, try several small logit steps, and accept the candidate with the best PPO clipped surrogate.

In [ ]:
def softmax_w(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - np.max(z))
    return e / e.sum()

rng_w = np.random.default_rng(0)
logits_train_w = np.array([0.0, 0.0])
history_w = []
for update_w in range(18):
    old_policy_w = softmax_w(logits_train_w)
    acts_w = rng_w.choice(2, size=80, p=old_policy_w)
    rewards_batch_w = (acts_w == 0).astype(float)
    adv_batch_w = rewards_batch_w - rewards_batch_w.mean()
    grad_logp_w = np.column_stack([(acts_w == 0).astype(float) - old_policy_w[0],
                                   (acts_w == 1).astype(float) - old_policy_w[1]])
    direction_w = (grad_logp_w * adv_batch_w[:, None]).mean(axis=0)
    scales_w = np.linspace(0.0, 2.0, 21)
    scores_w = []
    for scale_w in scales_w:
        cand_policy_w = softmax_w(logits_train_w + scale_w * direction_w)
        ratios_loop_w = cand_policy_w[acts_w] / old_policy_w[acts_w]
        clipped_loop_w = np.clip(ratios_loop_w, 0.8, 1.2)
        scores_w.append(float(np.mean(np.minimum(ratios_loop_w * adv_batch_w, clipped_loop_w * adv_batch_w))))
    best_scale_w = float(scales_w[int(np.argmax(scores_w))])
    logits_train_w = logits_train_w + best_scale_w * direction_w
    history_w.append(softmax_w(logits_train_w)[0])
print("first five p(a0):", np.round(history_w[:5], 3))
print("final p(a0):", round(float(history_w[-1]), 3))
assert history_w[-1] > history_w[0]

▶ What you'll see: the probability of the rewarding action rises over updates.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(history_w, marker="o", color="purple")
plt.axhline(0.5, color="gray", linestyle="--", label="initial random policy")
plt.ylim(0.45, 1.0)
plt.title("5: PPO-style updates improve a toy policy")
plt.xlabel("policy update")
plt.ylabel("π(action 0)")
plt.legend()
plt.show()

▶ What you'll see: a monotone-ish rise from about 0.5 toward a stronger preference for the rewarding action.

*Why it's done this way:* The toy loop separates data collection from policy improvement, exactly as PPO does. Advantages say which sampled actions deserved more probability, ratios measure how far the candidate moves from the data policy, and clipping prevents the search over logit steps from over-crediting a jump that the old batch cannot reliably justify.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each uses small
> numbers, prints every intermediate value with an inline `# ->`, draws one picture, and ends
> with an `assert`. Run them top to bottom.

### ✍️ Toy 1 · Softmax probabilities produce advantages

A policy turns logits into action probabilities; advantages compare each action's reward with the policy baseline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_seed = 0
t1_rng = np.random.default_rng(t1_seed)
print("rng seed:", t1_seed)  # -> 0
t1_logits = np.array([0.0, 1.0, -1.0])
print("logits:", t1_logits.tolist())  # -> [0.0, 1.0, -1.0]
t1_shifted = t1_logits - np.max(t1_logits)
print("shifted logits:", t1_shifted.tolist())  # -> [-1.0, 0.0, -2.0]
t1_exp = np.exp(t1_shifted)
print("exp scores:", np.round(t1_exp, 3).tolist())  # -> [0.368, 1.0, 0.135]
t1_policy = t1_exp / np.sum(t1_exp)
print("policy:", np.round(t1_policy, 3).tolist())  # -> [0.245, 0.665, 0.09]
t1_rewards = np.array([0.0, 2.0, -1.0])
print("action rewards:", t1_rewards.tolist())  # -> [0.0, 2.0, -1.0]
t1_baseline = float(t1_policy @ t1_rewards)
print("policy baseline:", round(t1_baseline, 3))  # -> 1.24
t1_advantages = t1_rewards - t1_baseline
print("advantages:", np.round(t1_advantages, 3).tolist())  # -> [-1.24, 0.76, -2.24]

plt.figure(figsize=(4.8, 3.0))
plt.bar(["a0", "a1", "a2"], t1_advantages, color=["indianred", "seagreen", "indianred"])
plt.axhline(0.0, color="black", linewidth=0.8)
plt.title("Toy 1 · advantage subtracts baseline")
plt.ylabel("A(a)")
plt.show()
assert np.allclose(np.round(t1_policy, 3), [0.245, 0.665, 0.09])

▶ What you'll see: only the high-reward middle action has positive advantage under this policy.

### ✍️ Toy 2 · Importance ratios reweight old-policy samples

When data came from an old policy, the new policy's probability of the sampled action appears as a ratio.

In [ ]:
import numpy as np

t2_seed = 0
t2_rng = np.random.default_rng(t2_seed)
print("rng seed:", t2_seed)  # -> 0
t2_old_policy = np.array([0.5, 0.3, 0.2])
print("old policy:", t2_old_policy.tolist())  # -> [0.5, 0.3, 0.2]
t2_new_policy = np.array([0.4, 0.45, 0.15])
print("new policy:", t2_new_policy.tolist())  # -> [0.4, 0.45, 0.15]
t2_actions = np.array([0, 1, 1, 2])
print("sampled actions:", t2_actions.tolist())  # -> [0, 1, 1, 2]
t2_advantages = np.array([1.0, 0.5, -0.2, -1.0])
print("advantages:", t2_advantages.tolist())  # -> [1.0, 0.5, -0.2, -1.0]
t2_old_probs = t2_old_policy[t2_actions]
print("old action probs:", t2_old_probs.tolist())  # -> [0.5, 0.3, 0.3, 0.2]
t2_new_probs = t2_new_policy[t2_actions]
print("new action probs:", t2_new_probs.tolist())  # -> [0.4, 0.45, 0.45, 0.15]
t2_ratios = t2_new_probs / t2_old_probs
print("ratios:", np.round(t2_ratios, 3).tolist())  # -> [0.8, 1.5, 1.5, 0.75]
t2_terms = t2_ratios * t2_advantages
print("ratio times advantage:", np.round(t2_terms, 3).tolist())  # -> [0.8, 0.75, -0.3, -0.75]

plt.figure(figsize=(4.8, 3.0))
plt.scatter(t2_ratios, t2_advantages, s=90, color="teal")
plt.axvline(1.0, color="black", linewidth=0.8)
plt.axhline(0.0, color="black", linewidth=0.8)
plt.title("Toy 2 · ratio reweights samples")
plt.xlabel("π_new / π_old")
plt.ylabel("advantage")
plt.show()
assert np.allclose(np.round(t2_ratios, 3), [0.8, 1.5, 1.5, 0.75])

▶ What you'll see: actions made more likely by the new policy get ratios above 1.

### ✍️ Toy 3 · A KL trust region rejects large jumps

TRPO allows only candidate policies whose KL divergence from the data-collecting policy stays under a budget.

In [ ]:
import numpy as np

t3_seed = 0
t3_rng = np.random.default_rng(t3_seed)
print("rng seed:", t3_seed)  # -> 0
t3_old_policy = np.array([0.5, 0.3, 0.2])
print("old policy:", t3_old_policy.tolist())  # -> [0.5, 0.3, 0.2]
t3_candidate_p0 = np.array([0.35, 0.45, 0.55, 0.65, 0.75])
print("candidate p(a0):", t3_candidate_p0.tolist())  # -> [0.35, 0.45, 0.55, 0.65, 0.75]
t3_candidate_tail = 1.0 - t3_candidate_p0
t3_candidates = np.column_stack([t3_candidate_p0, 0.6 * t3_candidate_tail, 0.4 * t3_candidate_tail])
print("candidate policies:", np.round(t3_candidates, 3).tolist())  # -> [[0.35, 0.39, 0.26], [0.45, 0.33, 0.22], [0.55, 0.27, 0.18], [0.65, 0.21, 0.14], [0.75, 0.15, 0.1]]
t3_log_old = np.log(t3_old_policy)
t3_log_candidates = np.log(t3_candidates)
t3_kls = np.sum(t3_old_policy * (t3_log_old - t3_log_candidates), axis=1)
print("KL(old||candidate):", np.round(t3_kls, 4).tolist())  # -> [0.0472, 0.005, 0.005, 0.0472, 0.1438]
t3_delta = 0.03
print("KL budget:", t3_delta)  # -> 0.03
t3_allowed = t3_kls <= t3_delta
print("allowed flags:", t3_allowed.astype(int).tolist())  # -> [0, 1, 1, 0, 0]
t3_rewards = np.array([2.0, 1.0, 0.0])
print("surrogate rewards:", t3_rewards.tolist())  # -> [2.0, 1.0, 0.0]
t3_scores = t3_candidates @ t3_rewards
print("surrogate scores:", np.round(t3_scores, 3).tolist())  # -> [1.09, 1.23, 1.37, 1.51, 1.65]
t3_free_choice = float(t3_candidate_p0[np.argmax(t3_scores)])
print("best unconstrained p(a0):", t3_free_choice)  # -> 0.75
t3_trust_choice = float(t3_candidate_p0[t3_allowed][np.argmax(t3_scores[t3_allowed])])
print("best trust-region p(a0):", t3_trust_choice)  # -> 0.55

plt.figure(figsize=(5.0, 3.0))
plt.plot(t3_candidate_p0, t3_scores, marker="o", label="surrogate")
plt.fill_between(t3_candidate_p0, t3_scores.min(), t3_scores.max(), where=t3_allowed, alpha=0.2, color="seagreen", label="KL ≤ 0.03")
plt.axvline(t3_trust_choice, color="seagreen", linestyle="--", label="TRPO pick")
plt.title("Toy 3 · KL limits the update")
plt.xlabel("candidate p(a0)")
plt.ylabel("score")
plt.legend()
plt.show()
assert t3_trust_choice == 0.55 and t3_free_choice == 0.75

▶ What you'll see: the reward score wants `p(a0)=0.75`, but the KL budget stops at `0.55`.

### ✍️ Toy 4 · PPO clipping flattens excessive ratios

PPO clips the probability ratio before multiplying by advantage, then keeps the conservative term.

In [ ]:
import numpy as np

t4_seed = 0
t4_rng = np.random.default_rng(t4_seed)
print("rng seed:", t4_seed)  # -> 0
t4_ratios = np.array([1.5, 0.5, 1.1, 0.9])
print("ratios:", t4_ratios.tolist())  # -> [1.5, 0.5, 1.1, 0.9]
t4_advantages = np.array([1.0, -1.0, 0.5, -0.5])
print("advantages:", t4_advantages.tolist())  # -> [1.0, -1.0, 0.5, -0.5]
t4_epsilon = 0.2
print("epsilon:", t4_epsilon)  # -> 0.2
t4_clipped_ratios = np.clip(t4_ratios, 1.0 - t4_epsilon, 1.0 + t4_epsilon)
print("clipped ratios:", t4_clipped_ratios.tolist())  # -> [1.2, 0.8, 1.1, 0.9]
t4_unclipped_terms = t4_ratios * t4_advantages
print("unclipped terms:", np.round(t4_unclipped_terms, 3).tolist())  # -> [1.5, -0.5, 0.55, -0.45]
t4_clipped_terms = t4_clipped_ratios * t4_advantages
print("clipped terms:", np.round(t4_clipped_terms, 3).tolist())  # -> [1.2, -0.8, 0.55, -0.45]
t4_ppo_terms = np.minimum(t4_unclipped_terms, t4_clipped_terms)
print("PPO terms:", np.round(t4_ppo_terms, 3).tolist())  # -> [1.2, -0.8, 0.55, -0.45]
t4_objective = float(np.mean(t4_ppo_terms))
print("mean PPO objective:", round(t4_objective, 3))  # -> 0.125

plt.figure(figsize=(5.0, 3.0))
plt.bar(np.arange(t4_ratios.size) - 0.15, t4_unclipped_terms, width=0.3, label="unclipped")
plt.bar(np.arange(t4_ratios.size) + 0.15, t4_ppo_terms, width=0.3, label="PPO")
plt.axhline(0.0, color="black", linewidth=0.8)
plt.title("Toy 4 · clipping chooses conservative terms")
plt.xlabel("sample")
plt.ylabel("term")
plt.legend()
plt.show()
assert np.allclose(np.round(t4_ppo_terms, 3), [1.2, -0.8, 0.55, -0.45])

▶ What you'll see: the too-large positive ratio is capped, and the too-small negative-advantage ratio is penalized.

### ✍️ Toy 5 · One PPO-style policy update chooses a safe step

A tiny batch gives a gradient direction, then PPO evaluates candidate logit steps with clipped ratios.

In [ ]:
import numpy as np

t5_seed = 0
t5_rng = np.random.default_rng(t5_seed)
print("rng seed:", t5_seed)  # -> 0
t5_logits = np.array([0.0, 0.0])
print("initial logits:", t5_logits.tolist())  # -> [0.0, 0.0]
t5_exp = np.exp(t5_logits - np.max(t5_logits))
t5_old_policy = t5_exp / np.sum(t5_exp)
print("old policy:", t5_old_policy.tolist())  # -> [0.5, 0.5]
t5_actions = t5_rng.choice(2, size=10, p=t5_old_policy)
print("sampled actions:", t5_actions.tolist())  # -> [1, 0, 0, 0, 1, 1, 1, 1, 1, 1]
t5_rewards = (t5_actions == 0).astype(float)
print("rewards:", t5_rewards.tolist())  # -> [0.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
t5_baseline = float(np.mean(t5_rewards))
print("batch baseline:", round(t5_baseline, 3))  # -> 0.3
t5_advantages = t5_rewards - t5_baseline
print("advantages:", np.round(t5_advantages, 3).tolist())  # -> [-0.3, 0.7, 0.7, 0.7, -0.3, -0.3, -0.3, -0.3, -0.3, -0.3]
t5_grad_logp = np.column_stack([(t5_actions == 0).astype(float) - t5_old_policy[0], (t5_actions == 1).astype(float) - t5_old_policy[1]])
print("first grad-logp row:", t5_grad_logp[0].tolist())  # -> [-0.5, 0.5]
t5_direction = np.mean(t5_grad_logp * t5_advantages[:, None], axis=0)
print("policy-gradient direction:", np.round(t5_direction, 3).tolist())  # -> [0.21, -0.21]
t5_scales = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
print("candidate scales:", t5_scales.tolist())  # -> [0.0, 0.5, 1.0, 1.5, 2.0]
t5_scores = []
t5_p0_values = []
for t5_scale in t5_scales:
    t5_candidate_logits = t5_logits + t5_scale * t5_direction
    t5_candidate_exp = np.exp(t5_candidate_logits - np.max(t5_candidate_logits))
    t5_candidate_policy = t5_candidate_exp / np.sum(t5_candidate_exp)
    t5_ratios = t5_candidate_policy[t5_actions] / t5_old_policy[t5_actions]
    t5_clipped = np.clip(t5_ratios, 0.8, 1.2)
    t5_score = float(np.mean(np.minimum(t5_ratios * t5_advantages, t5_clipped * t5_advantages)))
    t5_scores.append(t5_score)
    t5_p0_values.append(float(t5_candidate_policy[0]))
print("candidate p(a0):", np.round(t5_p0_values, 3).tolist())  # -> [0.5, 0.552, 0.603, 0.652, 0.698]
print("candidate scores:", np.round(t5_scores, 4).tolist())  # -> [-0.0, 0.0439, 0.084, 0.084, 0.084]
t5_best_index = int(np.argmax(t5_scores))
print("best scale:", float(t5_scales[t5_best_index]))  # -> 1.0
t5_new_p0 = t5_p0_values[t5_best_index]
print("new p(a0):", round(t5_new_p0, 3))  # -> 0.603

plt.figure(figsize=(5.0, 3.0))
plt.plot(t5_scales, t5_scores, marker="o", color="purple")
plt.axvline(t5_scales[t5_best_index], color="black", linestyle="--")
plt.title("Toy 5 · clipped surrogate picks a step")
plt.xlabel("logit step scale")
plt.ylabel("PPO score")
plt.show()
assert round(float(t5_new_p0), 3) == 0.603 and t5_new_p0 > t5_old_policy[0]

▶ What you'll see: the chosen clipped step raises the probability of the rewarding action from `0.5` to about `0.603`.

## 🛠️ Setup

In [ ]:
import numpy as np # Load NumPy for arrays, softmax policies, sampling, KL calculations, and small optimization loops.
import matplotlib.pyplot as plt # Load Matplotlib for the line plots, bar charts, and probability visualizations.
np.random.seed(0) # Fix the global random seed so every stochastic example is repeatable.

def softmax(z): # Convert logits into a probability distribution over actions.
    z = np.asarray(z, dtype=float) # Make the input numeric and predictable.
    e = np.exp(z - np.max(z)) # Subtract the max for numerical stability before exponentiating.
    return e / e.sum() # Normalize exponentials so probabilities sum to one.

def kl_div(p, q): # Compute KL(p||q) for two discrete action distributions.
    p = np.asarray(p, dtype=float) # Convert the first distribution to floats.
    q = np.asarray(q, dtype=float) # Convert the second distribution to floats.
    return float(np.sum(p * (np.log(p) - np.log(q)))) # Sum p log(p/q), assuming positive probabilities.

def ppo_terms(ratios, advantages, eps=0.2): # Compute PPO's clipped surrogate term per sample.
    clipped = np.clip(ratios, 1 - eps, 1 + eps) # Limit probability ratios to the trusted band.
    return np.minimum(ratios * advantages, clipped * advantages) # Use the conservative term sample by sample.

def discounted_return(rewards, gamma): # Compute G = sum_t gamma^t r_t for a finite reward sequence.
    rewards = np.asarray(rewards, dtype=float) # Convert rewards to a numeric vector.
    powers = gamma ** np.arange(len(rewards)) # Create discount powers 1, gamma, gamma^2, ... .
    return float(np.sum(powers * rewards)) # Return the discounted sum.

## 🟢 Basics (warm-up)

### Basic 1 — Softmax turns logits into a policy

**Goal.** Convert two action logits into probabilities, because TRPO and PPO update stochastic policies rather than deterministic action labels. We build it in 2 steps.

In [ ]:
logits_b1 = np.array([1.0, 0.0]) # Define preferences for two actions before normalization.
exp_b1 = np.exp(logits_b1 - np.max(logits_b1)) # Exponentiate shifted logits for a stable softmax calculation.
print("shifted exponentials:", np.round(exp_b1, 3)) # Inspect the positive weights before normalization.

▶ What you'll see: the larger logit produces the larger exponential weight.

In [ ]:
pi_b1 = softmax(logits_b1) # Convert logits to action probabilities.
print("policy probabilities:", np.round(pi_b1, 3)) # Inspect the stochastic policy.
assert np.allclose(np.round(pi_b1, 3), [0.731, 0.269]) # Verify the canonical softmax numbers.
plt.figure(figsize=(4, 3)) # Create a compact policy bar chart.
plt.bar(["action 0", "action 1"], pi_b1, color="teal") # Draw one bar per action probability.
plt.title("Basic 1: softmax policy") # Title the plot.
plt.ylabel("probability") # Label the y-axis.
plt.ylim(0, 1) # Keep probability scale fixed.
plt.show() # Display the policy.

▶ What you'll see: action 0 receives about 73.1% probability and action 1 receives about 26.9%.

👀 Takeaway: PPO and TRPO move logits, but the object they must control is the resulting probability distribution.

### Basic 2 — Expected reward under a policy

**Goal.** Compute a policy's expected one-step reward, because policy improvement means moving probability mass toward better consequences. We build it in 2 steps.

In [ ]:
pi_b2 = np.array([0.731, 0.269]) # Use the rounded policy from the softmax example.
rewards_b2 = np.array([2.0, 0.0]) # Define the reward received by each action in a one-state toy problem.
weighted_b2 = pi_b2 * rewards_b2 # Multiply each reward by the probability of taking that action.
print("weighted rewards:", np.round(weighted_b2, 3)) # Inspect each action's contribution.

▶ What you'll see: action 0 supplies all expected reward because action 1 pays zero.

In [ ]:
expected_b2 = float(np.sum(weighted_b2)) # Sum probability-weighted rewards into V(s).
print("expected reward:", round(expected_b2, 3)) # Inspect the policy value.
assert round(expected_b2, 3) == 1.462 # Verify 0.731*2 + 0.269*0.
plt.figure(figsize=(4, 3)) # Create a contribution chart.
plt.bar(["a0", "a1"], weighted_b2, color="seagreen") # Draw expected-reward contributions.
plt.title("Basic 2: probability-weighted reward") # Title the plot.
plt.ylabel("π(a)R(a)") # Label the contribution scale.
plt.show() # Display the chart.

▶ What you'll see: the expected reward is 1.462, below the best reward because the policy is still stochastic.

👀 Takeaway: changing action probability changes expected consequence, which is the bridge from policy logits to RL objective value.

### Basic 3 — Discounted return values delayed reward

**Goal.** Compute a finite discounted return, because RL rewards often arrive after the action that caused them. We build it in 2 steps.

In [ ]:
rewards_b3 = np.array([1.0, 0.0, 2.0]) # Define a three-step reward stream.
gamma_b3 = 0.9 # Set the discount so future rewards count but count less.
powers_b3 = gamma_b3 ** np.arange(len(rewards_b3)) # Create the discount multipliers.
print("discount powers:", np.round(powers_b3, 3)) # Inspect 1, gamma, gamma squared.

▶ What you'll see: the reward two steps away is multiplied by 0.81.

In [ ]:
G_b3 = discounted_return(rewards_b3, gamma_b3) # Compute G = r0 + gamma r1 + gamma^2 r2.
print("discounted return:", round(G_b3, 3)) # Inspect the delayed-consequence total.
assert round(G_b3, 3) == 2.62 # Verify 1 + 0.9*0 + 0.9^2*2.
plt.figure(figsize=(4, 3)) # Create a compact reward contribution plot.
plt.bar(["t0", "t1", "t2"], powers_b3 * rewards_b3, color="orange") # Show each discounted reward term.
plt.title("Basic 3: discounted reward terms") # Title the plot.
plt.ylabel("γ^t r_t") # Label the discounted contribution.
plt.show() # Display the chart.

▶ What you'll see: the delayed reward still matters, but it contributes 1.62 rather than 2.0.

👀 Takeaway: return is not immediate reward; it is the discounted ledger of consequences produced by actions.

### Basic 4 — Advantage subtracts a baseline

**Goal.** Convert action values into advantages, because policy gradients use relative usefulness rather than raw reward alone. We build it in 2 steps.

In [ ]:
q_b4 = np.array([2.0, 0.0]) # Define action values for two possible actions.
pi_b4 = softmax(np.array([1.0, 0.0])) # Define the current stochastic policy.
value_b4 = float(pi_b4 @ q_b4) # Compute V(s) as the policy-weighted action value.
print("state value baseline:", round(value_b4, 3)) # Inspect the baseline being subtracted.

▶ What you'll see: the baseline is the current policy's expected value, not a constant chosen by hand.

In [ ]:
adv_b4 = q_b4 - value_b4 # Compute A(s,a)=Q(s,a)-V(s).
print("advantages:", np.round(adv_b4, 3)) # Inspect positive and negative relative value.
assert np.allclose(np.round(adv_b4, 3), [0.538, -1.462]) # Verify the advantage numbers.
plt.figure(figsize=(4, 3)) # Create an advantage bar chart.
plt.bar(["a0", "a1"], adv_b4, color=["seagreen", "indianred"]) # Plot relative action quality.
plt.axhline(0, color="black", linewidth=0.8) # Add zero to separate good from bad.
plt.title("Basic 4: advantage signs") # Title the chart.
plt.ylabel("A(s,a)") # Label the advantage axis.
plt.show() # Display the chart.

▶ What you'll see: the better-than-baseline action has positive advantage and the worse action has negative advantage.

👀 Takeaway: advantage tells PPO which sampled actions should become more or less likely.

### Basic 5 — Log-probability gradient for one sample

**Goal.** Compute the gradient of log softmax for a sampled action, because policy gradient updates are built from $\nabla\log\pi(a\mid s)A$. We build it in 2 steps.

In [ ]:
pi_b5 = softmax(np.array([0.4, -0.2])) # Define a two-action policy.
action_b5 = 0 # Choose the sampled action whose log probability was observed.
one_hot_b5 = np.eye(2)[action_b5] # Create a one-hot vector for the sampled action.
print("policy:", np.round(pi_b5, 3), "one-hot action:", one_hot_b5) # Inspect inputs to the gradient.

▶ What you'll see: the one-hot action has 1 at the sampled action and 0 elsewhere.

In [ ]:
grad_logp_b5 = one_hot_b5 - pi_b5 # For softmax logits, gradient log pi(a) = one_hot(a) - pi.
print("grad log π(a):", np.round(grad_logp_b5, 3)) # Inspect the log-probability gradient.
assert abs(float(np.sum(grad_logp_b5))) < 1e-12 # Verify the gradient preserves probability mass.
plt.figure(figsize=(4, 3)) # Create a gradient bar chart.
plt.bar(["logit 0", "logit 1"], grad_logp_b5, color="purple") # Plot the push on each logit.
plt.axhline(0, color="black", linewidth=0.8) # Add a zero reference.
plt.title("Basic 5: log-probability gradient") # Title the plot.
plt.show() # Display the gradient.

▶ What you'll see: the sampled action's logit is pushed up while the other logit is pushed down.

👀 Takeaway: the policy-gradient direction is a probability-preserving tug on logits, scaled by advantage.

### Basic 6 — Probability ratio compares new and old policies

**Goal.** Compute PPO's ratio $r=\pi_{new}(a)/\pi_{old}(a)$, because old-policy samples must be reweighted under candidate policies. We build it in 2 steps.

In [ ]:
old_pi_b6 = np.array([0.6, 0.4]) # Define the policy that generated the data.
new_pi_b6 = np.array([0.78, 0.22]) # Define a candidate updated policy.
actions_b6 = np.array([0, 1, 0, 1]) # Store sampled actions from the old policy.
print("old probs for actions:", old_pi_b6[actions_b6]) # Inspect behavior probabilities.

▶ What you'll see: each sample carries the probability it had under the behavior policy.

In [ ]:
ratios_b6 = new_pi_b6[actions_b6] / old_pi_b6[actions_b6] # Compute likelihood ratios sample by sample.
print("ratios:", np.round(ratios_b6, 3)) # Inspect how the candidate policy reweights data.
assert np.allclose(np.round(ratios_b6, 3), [1.3, 0.55, 1.3, 0.55]) # Verify the ratio values.
plt.figure(figsize=(4, 3)) # Create a ratio chart.
plt.bar(np.arange(len(ratios_b6)), ratios_b6, color="teal") # Plot one ratio per sample.
plt.axhline(1.0, color="black", linestyle="--") # Mark no probability change.
plt.title("Basic 6: new/old probability ratios") # Title the plot.
plt.ylabel("r") # Label ratio axis.
plt.show() # Display the ratios.

▶ What you'll see: action-0 samples became more likely and action-1 samples became less likely.

👀 Takeaway: ratios are the exact measurement PPO clips and TRPO indirectly constrains.

### Basic 7 — Unclipped surrogate term

**Goal.** Multiply ratios by advantages, because the vanilla policy surrogate rewards probability changes that align with advantage signs. We build it in 2 steps.

In [ ]:
ratios_b7 = np.array([1.3, 0.55, 1.1, 0.9]) # Define candidate probability ratios.
adv_b7 = np.array([1.0, -1.0, 0.4, -0.2]) # Define sample advantages.
print("ratios:", ratios_b7) # Inspect the probability changes.
print("advantages:", adv_b7) # Inspect the learning signal.

▶ What you'll see: samples include both positive and negative advantages.

In [ ]:
terms_b7 = ratios_b7 * adv_b7 # Compute the unclipped surrogate contribution per sample.
print("unclipped terms:", np.round(terms_b7, 3)) # Inspect the objective terms.
assert round(float(np.mean(terms_b7)), 3) == 0.253 # Verify the mean surrogate term.
plt.figure(figsize=(4, 3)) # Create a contribution chart.
plt.bar(np.arange(len(terms_b7)), terms_b7, color=["seagreen" if x >= 0 else "indianred" for x in terms_b7]) # Color by sign.
plt.axhline(0, color="black", linewidth=0.8) # Add zero reference.
plt.title("Basic 7: r × A terms") # Title the plot.
plt.ylabel("surrogate term") # Label term axis.
plt.show() # Display the chart.

▶ What you'll see: a high ratio helps positive advantage, while a low ratio helps negative advantage.

👀 Takeaway: the unclipped surrogate has no built-in brake on very large probability changes.

### Basic 8 — PPO clipping caps helpful ratios

**Goal.** Apply the PPO clipped objective, because huge ratio changes should stop improving the local surrogate. We build it in 2 steps.

In [ ]:
ratios_b8 = np.array([1.5, 1.1, 0.5, 0.9]) # Include ratios outside and inside the clip band.
adv_b8 = np.array([1.0, 1.0, -1.0, -1.0]) # Pair positive and negative advantages with those ratios.
eps_b8 = 0.2 # Use the common PPO clip width.
clipped_b8 = np.clip(ratios_b8, 1 - eps_b8, 1 + eps_b8) # Clip ratios into [0.8, 1.2].
print("clipped ratios:", clipped_b8) # Inspect the capped ratios.

▶ What you'll see: 1.5 becomes 1.2 and 0.5 becomes 0.8.

In [ ]:
ppo_b8 = ppo_terms(ratios_b8, adv_b8, eps_b8) # Compute conservative PPO terms.
print("PPO terms:", np.round(ppo_b8, 3)) # Inspect clipped objective values.
assert np.allclose(ppo_b8, [1.2, 1.1, -0.8, -0.9]) # Verify signs and caps.
plt.figure(figsize=(4, 3)) # Create a clipped-vs-unclipped chart.
plt.plot(ratios_b8 * adv_b8, "o", label="unclipped") # Plot raw rA terms.
plt.plot(ppo_b8, "s", label="PPO clipped") # Plot clipped terms.
plt.axhline(0, color="black", linewidth=0.8) # Add zero reference.
plt.title("Basic 8: clipped surrogate terms") # Title the plot.
plt.legend() # Show labels.
plt.show() # Display the comparison.

▶ What you'll see: only the terms whose ratios are too helpful get pulled back.

👀 Takeaway: clipping is asymmetric through the advantage sign; it blocks over-improvement, not every ratio outside the band equally.

### Basic 9 — KL divergence measures policy movement

**Goal.** Compute KL divergence between old and new action distributions, because TRPO uses KL as its trust-region distance. We build it in 2 steps.

In [ ]:
old_pi_b9 = np.array([0.6, 0.4]) # Define the data-collecting policy.
new_pi_b9 = np.array([0.78, 0.22]) # Define a candidate update.
pieces_b9 = old_pi_b9 * (np.log(old_pi_b9) - np.log(new_pi_b9)) # Compute per-action KL contributions.
print("KL pieces:", np.round(pieces_b9, 4)) # Inspect each action's contribution.

▶ What you'll see: one contribution can be negative, but the total KL is nonnegative.

In [ ]:
kl_b9 = kl_div(old_pi_b9, new_pi_b9) # Sum p log(p/q) over actions.
print("KL(old||new):", round(kl_b9, 4)) # Inspect policy movement.
assert round(kl_b9, 4) == 0.0817 # Verify the trust-region distance.
plt.figure(figsize=(4, 3)) # Create a KL contribution chart.
plt.bar(["a0", "a1"], pieces_b9, color="orange") # Plot per-action KL terms.
plt.axhline(0, color="black", linewidth=0.8) # Add zero reference.
plt.title("Basic 9: KL contribution pieces") # Title the plot.
plt.ylabel("p_old log(p_old/p_new)") # Label contribution axis.
plt.show() # Display the chart.

▶ What you'll see: the total KL is about 0.0817, larger than a typical small TRPO budget like 0.01 or 0.02.

👀 Takeaway: KL measures how far the entire distribution moved, not whether reward improved.

### Basic 10 — A tiny trust-region accept rule

**Goal.** Accept only candidate policies whose KL stays below a budget, because TRPO improves within a local neighborhood of the old policy. We build it in 3 steps.

In [ ]:
old_pi_b10 = np.array([0.6, 0.4]) # Define the old policy.
candidates_b10 = np.array([[0.62, 0.38], [0.70, 0.30], [0.85, 0.15]]) # Define candidate policies.
delta_b10 = 0.03 # Set a small KL budget.
kls_b10 = np.array([kl_div(old_pi_b10, q_b10) for q_b10 in candidates_b10]) # Compute KL for each candidate.
print("candidate KLs:", np.round(kls_b10, 4)) # Inspect trust-region distances.

▶ What you'll see: small probability changes have small KL and large jumps have larger KL.

In [ ]:
values_b10 = candidates_b10[:, 0] * 2.0 # Use expected reward when action 0 pays 2 and action 1 pays 0.
allowed_b10 = kls_b10 <= delta_b10 # Mark candidates that satisfy the constraint.
print("values:", np.round(values_b10, 3), "allowed:", allowed_b10.astype(int)) # Inspect objective and constraint.

In [ ]:
best_idx_b10 = np.where(allowed_b10)[0][np.argmax(values_b10[allowed_b10])] # Select best allowed candidate.
print("chosen candidate:", candidates_b10[best_idx_b10]) # Inspect the trust-region choice.
assert np.allclose(candidates_b10[best_idx_b10], [0.70, 0.30]) # Verify the best safe update.
plt.figure(figsize=(4, 3)) # Create a constrained-choice plot.
plt.scatter(kls_b10, values_b10, s=100, color=["seagreen" if ok else "indianred" for ok in allowed_b10]) # Plot value vs KL.
plt.axvline(delta_b10, color="black", linestyle="--", label="δ") # Mark the KL budget.
plt.title("Basic 10: best value inside KL budget") # Title the plot.
plt.xlabel("KL(old||new)") # Label the constraint axis.
plt.ylabel("surrogate value") # Label the objective axis.
plt.legend() # Show the budget label.
plt.show() # Display the selection.

▶ What you'll see: the largest reward candidate is rejected if it sits outside the trust region.

👀 Takeaway: TRPO's core idea is not “take the best-looking step”; it is “take the best-looking step that remains close enough to trust.”

## 🟡 Easy

### Easy 1 — Compute returns for a short trajectory

**Goal.** Turn a trajectory's reward stream into return targets, because policy-gradient advantages need estimates of consequence after each time step. We build it in 3 steps.

In [ ]:
rewards_e1 = np.array([1.0, 0.0, 2.0, 1.0]) # Define rewards along one short trajectory.
gamma_e1 = 0.9 # Set the discount factor.
returns_e1 = np.zeros_like(rewards_e1) # Allocate return values for each time step.
print("rewards:", rewards_e1) # Inspect the trajectory rewards.

▶ What you'll see: rewards arrive at several different times, not just immediately.

In [ ]:
running_e1 = 0.0 # Start the backward return accumulator after the final step.
for t_e1 in range(len(rewards_e1) - 1, -1, -1): # Walk backward through the trajectory.
    running_e1 = rewards_e1[t_e1] + gamma_e1 * running_e1 # Apply G_t = r_t + gamma G_{t+1}.
    returns_e1[t_e1] = running_e1 # Store the return for this time step.
print("returns:", np.round(returns_e1, 3)) # Inspect every discounted consequence target.
assert np.allclose(np.round(returns_e1, 3), [3.349, 2.61, 2.9, 1.0]) # Verify the return recursion.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a reward-vs-return plot.
plt.plot(rewards_e1, marker="o", label="reward") # Plot immediate rewards.
plt.plot(returns_e1, marker="s", label="return") # Plot discounted future consequence.
plt.title("Easy 1: rewards versus returns") # Title the comparison.
plt.xlabel("time step") # Label time axis.
plt.legend() # Show series labels.
plt.show() # Display the plot.

▶ What you'll see: returns are smoother and often larger than immediate rewards because they include future payoff.

👀 Takeaway: PPO can use immediate rewards only after they have been converted into return or advantage estimates.

### Easy 2 — Estimate advantages from a value baseline

**Goal.** Subtract a value baseline from returns, because advantage estimates reduce policy-gradient variance. We build it in 3 steps.

In [ ]:
returns_e2 = np.array([3.349, 2.610, 2.900, 1.000]) # Use return targets from the previous style of calculation.
values_e2 = np.array([2.5, 2.0, 2.2, 1.1]) # Define current value estimates for the visited states.
adv_e2 = returns_e2 - values_e2 # Compute A_t = G_t - V(s_t).
print("raw advantages:", np.round(adv_e2, 3)) # Inspect above- and below-baseline outcomes.

▶ What you'll see: most outcomes beat the baseline, while the final one slightly underperforms.

In [ ]:
adv_norm_e2 = (adv_e2 - adv_e2.mean()) / (adv_e2.std() + 1e-8) # Normalize advantages for a steadier update scale.
print("normalized advantages:", np.round(adv_norm_e2, 3)) # Inspect the centered/scaled learning signal.
assert round(float(adv_e2[0]), 3) == 0.849 # Verify the first advantage.

In [ ]:
plt.figure(figsize=(5, 3)) # Create an advantage comparison plot.
plt.bar(np.arange(len(adv_e2)) - 0.18, adv_e2, width=0.36, label="raw", color="teal") # Plot raw advantages.
plt.bar(np.arange(len(adv_norm_e2)) + 0.18, adv_norm_e2, width=0.36, label="normalized", color="orange") # Plot normalized advantages.
plt.axhline(0, color="black", linewidth=0.8) # Add zero reference.
plt.title("Easy 2: advantage estimates") # Title the chart.
plt.legend() # Show labels.
plt.show() # Display the chart.

▶ What you'll see: normalization changes scale but preserves which samples are above or below the batch average.

👀 Takeaway: a baseline changes variance and scale, not the basic idea that positive advantage increases sampled action probability.

### Easy 3 — Compare TRPO and PPO brakes on one batch

**Goal.** Evaluate the same candidate update with KL and clipping, because TRPO and PPO are two ways to limit destructive policy jumps. We build it in 3 steps.

In [ ]:
old_pi_e3 = np.array([0.55, 0.45]) # Define the data-collecting policy.
cand_pi_e3 = np.array([0.72, 0.28]) # Define a candidate improved policy.
actions_e3 = np.array([0, 0, 1, 0, 1, 1]) # Store actions sampled from the old policy.
advantages_e3 = np.array([1.0, 0.6, -0.4, 0.8, -0.7, -0.2]) # Store estimated advantages.
ratios_e3 = cand_pi_e3[actions_e3] / old_pi_e3[actions_e3] # Compute candidate ratios.
print("ratios:", np.round(ratios_e3, 3)) # Inspect how far the candidate moved sampled actions.

▶ What you'll see: action 0 ratios are above 1 and action 1 ratios are below 1.

In [ ]:
kl_e3 = kl_div(old_pi_e3, cand_pi_e3) # Compute TRPO-style distribution movement.
ppo_mean_e3 = float(np.mean(ppo_terms(ratios_e3, advantages_e3, eps=0.2))) # Compute PPO clipped surrogate mean.
unclip_mean_e3 = float(np.mean(ratios_e3 * advantages_e3)) # Compute unclipped surrogate mean.
print("KL:", round(kl_e3, 4), "unclipped:", round(unclip_mean_e3, 3), "PPO:", round(ppo_mean_e3, 3)) # Inspect both brakes.
assert kl_e3 > 0.02 # Verify this candidate would fail a small TRPO budget.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a surrogate comparison plot.
plt.bar(["unclipped", "PPO clipped"], [unclip_mean_e3, ppo_mean_e3], color=["gray", "seagreen"]) # Compare objective estimates.
plt.title("Easy 3: clipping lowers an aggressive surrogate") # Title the plot.
plt.ylabel("mean surrogate") # Label objective scale.
plt.show() # Display the chart.

▶ What you'll see: the clipped objective is more conservative, while the KL tells you the full distribution moved a lot.

👀 Takeaway: TRPO constrains policy distance explicitly; PPO modifies the objective so aggressive ratios stop paying.

### Easy 4 — One PPO update on a sampled bandit batch

**Goal.** Run one policy update on sampled data, because PPO alternates between collecting old-policy actions and improving the policy locally. We build it in 4 steps.

In [ ]:
rng_e4 = np.random.default_rng(4) # Create reproducible local randomness.
logits_e4 = np.array([0.0, 0.0]) # Start from a uniform two-action policy.
old_pi_e4 = softmax(logits_e4) # Convert logits into old action probabilities.
actions_e4 = rng_e4.choice(2, size=40, p=old_pi_e4) # Sample actions from the old policy.
rewards_e4 = (actions_e4 == 0).astype(float) # Give reward 1 to action 0 and 0 to action 1.
print("old policy:", old_pi_e4, "mean reward:", round(float(rewards_e4.mean()), 3)) # Inspect collected batch quality.

▶ What you'll see: the initial random policy earns about half reward on average.

In [ ]:
adv_e4 = rewards_e4 - rewards_e4.mean() # Use the batch mean reward as a baseline.
grad_logp_e4 = np.column_stack([(actions_e4 == 0).astype(float) - old_pi_e4[0], (actions_e4 == 1).astype(float) - old_pi_e4[1]]) # Compute log-softmax gradients.
direction_e4 = (grad_logp_e4 * adv_e4[:, None]).mean(axis=0) # Average policy-gradient direction.
print("update direction:", np.round(direction_e4, 4)) # Inspect how logits should move.

In [ ]:
candidate_logits_e4 = logits_e4 + 1.0 * direction_e4 # Try one small step along the policy-gradient direction.
new_pi_e4 = softmax(candidate_logits_e4) # Convert candidate logits into probabilities.
ratios_e4 = new_pi_e4[actions_e4] / old_pi_e4[actions_e4] # Compute ratios for sampled actions.
objective_e4 = float(np.mean(ppo_terms(ratios_e4, adv_e4, eps=0.2))) # Compute the clipped PPO objective.
print("new policy:", np.round(new_pi_e4, 3), "PPO objective:", round(objective_e4, 4)) # Inspect the candidate update.
assert new_pi_e4[0] > old_pi_e4[0] # Verify reward action became more likely.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a before-after policy chart.
plt.bar(["old a0", "new a0"], [old_pi_e4[0], new_pi_e4[0]], color=["gray", "purple"]) # Compare probability of rewarding action.
plt.ylim(0, 1) # Keep probability scale fixed.
plt.title("Easy 4: one PPO-style policy step") # Title the plot.
plt.ylabel("π(action 0)") # Label the probability axis.
plt.show() # Display the chart.

▶ What you'll see: the probability of the rewarding action increases after one advantage-weighted step.

👀 Takeaway: even a tiny PPO-style loop is data collection, advantage estimation, ratio computation, and a conservative policy move.

### Easy 5 — Visualize clip width as a hyperparameter

**Goal.** Compare different PPO clip widths, because epsilon controls how much probability-ratio movement still receives objective credit. We build it in 3 steps.

In [ ]:
ratio_grid_e5 = np.linspace(0.4, 1.6, 121) # Sweep possible probability ratios.
eps_values_e5 = [0.1, 0.2, 0.3] # Compare three clip widths.
adv_e5 = 1.0 # Use a positive advantage so the upper clip is easiest to see.
print("epsilon values:", eps_values_e5) # Inspect the hyperparameters.

▶ What you'll see: smaller epsilon means a narrower trusted ratio band.

In [ ]:
curves_e5 = [] # Store clipped objective curves.
for eps_e5 in eps_values_e5: # Loop over clip widths.
    curves_e5.append(ppo_terms(ratio_grid_e5, adv_e5 * np.ones_like(ratio_grid_e5), eps=eps_e5)) # Compute clipped positive-advantage curve.
print("value at r=1.5:", [round(float(c_e5[-11]), 3) for c_e5 in curves_e5]) # Inspect near r=1.5 for each epsilon.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a clip-width comparison plot.
for eps_e5, curve_e5 in zip(eps_values_e5, curves_e5): # Plot each epsilon curve.
    plt.plot(ratio_grid_e5, curve_e5, label=f"eps={eps_e5}") # Draw clipped surrogate for this width.
plt.plot(ratio_grid_e5, ratio_grid_e5, color="gray", linestyle="--", label="unclipped") # Add unclipped reference.
plt.title("Easy 5: clip width controls the flat region") # Title the plot.
plt.xlabel("ratio r") # Label ratio axis.
plt.ylabel("clipped term for A=+1") # Label objective axis.
plt.legend() # Show epsilon labels.
plt.show() # Display the plot.

▶ What you'll see: smaller epsilon flattens sooner, while larger epsilon allows more probability movement before clipping.

👀 Takeaway: PPO's epsilon is a trust-region-like knob implemented directly in the surrogate objective.

## 🔴 Advanced

### Advanced 1 — Grid-search a TRPO-style constrained update

**Goal.** Choose the best candidate policy under a KL budget, because TRPO is constrained optimization rather than unconstrained hill climbing. We build it in 4 steps.

In [ ]:
old_pi_a1 = np.array([0.55, 0.45]) # Define the old policy that collected data.
ps_a1 = np.linspace(0.50, 0.95, 46) # Sweep candidate probabilities for action 0.
candidates_a1 = np.column_stack([ps_a1, 1 - ps_a1]) # Build two-action distributions from the sweep.
delta_a1 = 0.015 # Set a small trust-region budget.
print("number of candidates:", len(candidates_a1)) # Inspect grid resolution.

▶ What you'll see: the search tries many possible probability moves from the old policy.

In [ ]:
kls_a1 = np.array([kl_div(old_pi_a1, q_a1) for q_a1 in candidates_a1]) # Compute KL(old||candidate) for every candidate.
values_a1 = candidates_a1[:, 0] * 1.5 + candidates_a1[:, 1] * 0.2 # Compute a toy surrogate where action 0 is better.
allowed_a1 = kls_a1 <= delta_a1 # Apply the trust-region constraint.
print("allowed candidates:", int(allowed_a1.sum())) # Inspect how many candidates survive.

In [ ]:
best_allowed_idx_a1 = np.where(allowed_a1)[0][np.argmax(values_a1[allowed_a1])] # Select best objective among allowed candidates.
best_free_idx_a1 = int(np.argmax(values_a1)) # Select unconstrained best objective.
print("TRPO-like p(a0):", round(float(candidates_a1[best_allowed_idx_a1, 0]), 3)) # Inspect constrained choice.
print("unconstrained p(a0):", round(float(candidates_a1[best_free_idx_a1, 0]), 3)) # Inspect greedy choice.
assert candidates_a1[best_allowed_idx_a1, 0] < candidates_a1[best_free_idx_a1, 0] # Verify the constraint restrains the jump.

In [ ]:
plt.figure(figsize=(5, 3)) # Create value-vs-KL visualization.
plt.scatter(kls_a1, values_a1, c=np.where(allowed_a1, "seagreen", "indianred"), s=35) # Color feasible and infeasible policies.
plt.axvline(delta_a1, color="black", linestyle="--", label="δ") # Mark the trust-region budget.
plt.scatter([kls_a1[best_allowed_idx_a1]], [values_a1[best_allowed_idx_a1]], s=140, facecolors="none", edgecolors="blue", label="chosen") # Circle selected candidate.
plt.title("Advanced 1: constrained policy search") # Title the plot.
plt.xlabel("KL(old||candidate)") # Label constraint axis.
plt.ylabel("surrogate value") # Label objective axis.
plt.legend() # Show labels.
plt.show() # Display the constrained search.

▶ What you'll see: the chosen point is the highest-value green point, not the highest-value point overall.

👀 Takeaway: TRPO spends a KL budget to buy improvement while staying inside the region where old-policy data is credible.

### Advanced 2 — Train PPO for several updates on a toy bandit

**Goal.** Implement repeated PPO-style updates from scratch, because the algorithm's behavior is easiest to trust when every ratio and clip is visible. We build it in 5 steps.

In [ ]:
rng_a2 = np.random.default_rng(12) # Create reproducible randomness for the toy bandit.
logits_a2 = np.array([0.0, 0.0]) # Start from an indifferent policy.
prob_history_a2 = [] # Store probability of the rewarding action after each update.
kl_history_a2 = [] # Store KL from old to new policy after each accepted update.
print("initial policy:", softmax(logits_a2)) # Inspect the starting distribution.

▶ What you'll see: the policy starts at 0.5/0.5.

In [ ]:
for update_a2 in range(25): # Run repeated PPO-style policy updates.
    old_pi_a2 = softmax(logits_a2) # Freeze the data-collecting policy.
    actions_a2 = rng_a2.choice(2, size=100, p=old_pi_a2) # Collect a batch of old-policy actions.
    rewards_a2 = (actions_a2 == 0).astype(float) # Reward action 0 in the toy environment.
    adv_a2 = rewards_a2 - rewards_a2.mean() # Estimate advantages with a batch-mean baseline.
    grad_logp_a2 = np.column_stack([(actions_a2 == 0).astype(float) - old_pi_a2[0], (actions_a2 == 1).astype(float) - old_pi_a2[1]]) # Compute log-probability gradients.
    direction_a2 = (grad_logp_a2 * adv_a2[:, None]).mean(axis=0) # Average advantage-weighted direction.
    scales_a2 = np.linspace(0.0, 2.5, 26) # Try a small line search over step scales.
    objectives_a2 = [] # Store clipped objectives for candidates.
    policies_a2 = [] # Store candidate policies.
    for scale_a2 in scales_a2: # Score each candidate scale.
        cand_pi_a2 = softmax(logits_a2 + scale_a2 * direction_a2) # Candidate policy.
        ratios_a2 = cand_pi_a2[actions_a2] / old_pi_a2[actions_a2] # Candidate ratios on old samples.
        objectives_a2.append(float(np.mean(ppo_terms(ratios_a2, adv_a2, eps=0.2)))) # Candidate PPO objective.
        policies_a2.append(cand_pi_a2) # Save candidate distribution.
    chosen_a2 = int(np.argmax(objectives_a2)) # Pick best clipped objective.
    new_pi_a2 = policies_a2[chosen_a2] # Read the accepted candidate policy.
    logits_a2 = np.log(new_pi_a2) # Store equivalent logits for the next update.
    prob_history_a2.append(new_pi_a2[0]) # Track improvement.
    kl_history_a2.append(kl_div(old_pi_a2, new_pi_a2)) # Track policy movement.
print("final policy:", np.round(softmax(logits_a2), 3)) # Inspect trained policy.
assert prob_history_a2[-1] > 0.8 # Verify PPO learned to favor action 0.

In [ ]:
print("first probabilities:", np.round(prob_history_a2[:5], 3)) # Inspect early learning.
print("mean KL per update:", round(float(np.mean(kl_history_a2)), 4)) # Inspect update size.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3)) # Create side-by-side diagnostics.
ax[0].plot(prob_history_a2, marker="o", color="purple") # Plot policy improvement.
ax[0].set_title("PPO toy policy improves") # Title first panel.
ax[0].set_xlabel("update") # Label x-axis.
ax[0].set_ylabel("π(action 0)") # Label probability axis.
ax[0].set_ylim(0.45, 1.0) # Keep probability range readable.
ax[1].plot(kl_history_a2, marker="s", color="orange") # Plot KL movement per update.
ax[1].set_title("KL remains small") # Title second panel.
ax[1].set_xlabel("update") # Label x-axis.
ax[1].set_ylabel("KL old||new") # Label KL axis.
plt.tight_layout() # Reduce overlap.
plt.show() # Display diagnostics.

▶ What you'll see: action-0 probability rises while individual KL jumps stay modest.

👀 Takeaway: PPO is still policy gradient, but clipping makes each reuse of old data less willing to endorse a giant probability leap.

### Advanced 3 — Show how clipping handles negative advantages

**Goal.** Inspect negative-advantage samples separately, because PPO's conservative objective is easiest to misunderstand there. We build it in 4 steps.

In [ ]:
ratios_a3 = np.linspace(0.4, 1.6, 13) # Sweep ratios across and beyond the clip range.
adv_pos_a3 = np.ones_like(ratios_a3) # Positive advantage case.
adv_neg_a3 = -np.ones_like(ratios_a3) # Negative advantage case.
eps_a3 = 0.2 # Set PPO clip width.
print("ratios:", np.round(ratios_a3, 2)) # Inspect ratio grid.

▶ What you'll see: the grid includes aggressive decreases and increases in action probability.

In [ ]:
term_pos_a3 = ppo_terms(ratios_a3, adv_pos_a3, eps_a3) # Compute clipped terms for positive advantages.
term_neg_a3 = ppo_terms(ratios_a3, adv_neg_a3, eps_a3) # Compute clipped terms for negative advantages.
print("positive terms:", np.round(term_pos_a3, 2)) # Inspect upper clipping.
print("negative terms:", np.round(term_neg_a3, 2)) # Inspect lower clipping.

In [ ]:
idx_low_a3 = int(np.where(np.isclose(ratios_a3, 0.5))[0][0]) # Locate ratio 0.5.
idx_high_a3 = int(np.where(np.isclose(ratios_a3, 1.5))[0][0]) # Locate ratio 1.5.
print("A=-1 at r=0.5:", term_neg_a3[idx_low_a3], "A=+1 at r=1.5:", term_pos_a3[idx_high_a3]) # Inspect key clipped values.
assert term_neg_a3[idx_low_a3] == -0.8 and term_pos_a3[idx_high_a3] == 1.2 # Verify asymmetric caps.

In [ ]:
plt.figure(figsize=(5, 3)) # Create positive/negative clipping plot.
plt.plot(ratios_a3, term_pos_a3, marker="o", label="A=+1", color="seagreen") # Plot positive advantage curve.
plt.plot(ratios_a3, term_neg_a3, marker="s", label="A=-1", color="indianred") # Plot negative advantage curve.
plt.axvspan(1 - eps_a3, 1 + eps_a3, alpha=0.12, color="steelblue") # Show trusted band.
plt.title("Advanced 3: clipping depends on advantage sign") # Title the plot.
plt.xlabel("ratio r") # Label ratio axis.
plt.ylabel("PPO term") # Label objective axis.
plt.legend() # Show labels.
plt.show() # Display the plot.

▶ What you'll see: for negative advantages, making the bad action too unlikely stops improving once r drops below 0.8.

👀 Takeaway: PPO clipping is conservative because it blocks whichever ratio direction would make the sample look too good.

### Advanced 4 — Compare clip width and empirical KL during training

**Goal.** Train the same toy policy with several clip widths, because PPO's epsilon affects both learning speed and policy movement. We build it in 5 steps.

In [ ]:
eps_grid_a4 = np.array([0.05, 0.2, 0.4]) # Compare narrow, standard, and wide clipping.
final_probs_a4 = [] # Store final action-0 probabilities.
mean_kls_a4 = [] # Store mean KL per update.
print("clip widths:", eps_grid_a4) # Inspect sweep values.

▶ What you'll see: the experiment changes only epsilon.

In [ ]:
for eps_a4 in eps_grid_a4: # Train one policy per clip width.
    rng_a4 = np.random.default_rng(44) # Reset randomness for fair comparison.
    logits_a4 = np.array([0.0, 0.0]) # Reset policy.
    kls_run_a4 = [] # Store KLs for this run.
    for update_a4 in range(16): # Run a short PPO loop.
        old_pi_a4 = softmax(logits_a4) # Freeze old policy.
        actions_a4 = rng_a4.choice(2, size=80, p=old_pi_a4) # Sample a batch.
        rewards_a4 = (actions_a4 == 0).astype(float) # Reward action 0.
        adv_a4 = rewards_a4 - rewards_a4.mean() # Batch-mean baseline.
        grad_logp_a4 = np.column_stack([(actions_a4 == 0).astype(float) - old_pi_a4[0], (actions_a4 == 1).astype(float) - old_pi_a4[1]]) # Log-policy gradients.
        direction_a4 = (grad_logp_a4 * adv_a4[:, None]).mean(axis=0) # Policy-gradient direction.
        scales_a4 = np.linspace(0.0, 3.0, 31) # Candidate line-search scales.
        best_obj_a4 = -1e9 # Initialize best objective.
        best_pi_a4 = old_pi_a4 # Initialize best policy.
        for scale_a4 in scales_a4: # Score each candidate.
            cand_pi_a4 = softmax(logits_a4 + scale_a4 * direction_a4) # Candidate policy.
            ratios_a4 = cand_pi_a4[actions_a4] / old_pi_a4[actions_a4] # Probability ratios.
            obj_a4 = float(np.mean(ppo_terms(ratios_a4, adv_a4, eps=eps_a4))) # Clipped objective for this epsilon.
            if obj_a4 > best_obj_a4: # Keep best candidate.
                best_obj_a4 = obj_a4 # Update best score.
                best_pi_a4 = cand_pi_a4 # Update best policy.
        kls_run_a4.append(kl_div(old_pi_a4, best_pi_a4)) # Record update KL.
        logits_a4 = np.log(best_pi_a4) # Move to selected policy.
    final_probs_a4.append(softmax(logits_a4)[0]) # Store final rewarding-action probability.
    mean_kls_a4.append(float(np.mean(kls_run_a4))) # Store mean KL.
print("final probabilities:", np.round(final_probs_a4, 3)) # Inspect learning speed.
print("mean KLs:", np.round(mean_kls_a4, 4)) # Inspect movement sizes.

In [ ]:
assert final_probs_a4[0] <= final_probs_a4[-1] # Verify wider clipping learned at least as aggressively in this setup.
print("narrow vs wide final:", round(float(final_probs_a4[0]), 3), round(float(final_probs_a4[-1]), 3)) # Inspect comparison.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3)) # Create side-by-side sweep plots.
ax[0].bar([str(e) for e in eps_grid_a4], final_probs_a4, color="purple") # Plot final performance proxy.
ax[0].set_title("final π(action 0)") # Title first panel.
ax[0].set_xlabel("clip epsilon") # Label x-axis.
ax[0].set_ylim(0.5, 1.0) # Keep probability range readable.
ax[1].bar([str(e) for e in eps_grid_a4], mean_kls_a4, color="orange") # Plot mean policy movement.
ax[1].set_title("mean KL per update") # Title second panel.
ax[1].set_xlabel("clip epsilon") # Label x-axis.
plt.tight_layout() # Reduce overlap.
plt.show() # Display sweep.

▶ What you'll see: wider clips usually allow faster movement and larger KL, while narrow clips are more cautious.

👀 Takeaway: PPO's clip width is a safety-speed tradeoff, not a cosmetic constant.

### Advanced 5 — Detect a destructive probability jump

**Goal.** Compare a moderate update with an extreme update on the same batch, because the whole point of TRPO and PPO is to reject deceptively large jumps. We build it in 4 steps.

In [ ]:
old_pi_a5 = np.array([0.5, 0.5]) # Define the old data policy.
actions_a5 = np.array([0, 0, 0, 1, 1, 1]) # Create a balanced old-policy batch.
advantages_a5 = np.array([1.0, 0.8, 0.6, -0.6, -0.8, -1.0]) # Good action 0, bad action 1.
moderate_pi_a5 = np.array([0.65, 0.35]) # Define a plausible improved policy.
extreme_pi_a5 = np.array([0.98, 0.02]) # Define a destructive jump far from the data policy.
print("old/moderate/extreme:", old_pi_a5, moderate_pi_a5, extreme_pi_a5) # Inspect policies.

▶ What you'll see: the extreme policy nearly eliminates action 1.

In [ ]:
ratios_mod_a5 = moderate_pi_a5[actions_a5] / old_pi_a5[actions_a5] # Ratios for moderate update.
ratios_ext_a5 = extreme_pi_a5[actions_a5] / old_pi_a5[actions_a5] # Ratios for extreme update.
ppo_mod_a5 = float(np.mean(ppo_terms(ratios_mod_a5, advantages_a5, eps=0.2))) # PPO objective for moderate update.
ppo_ext_a5 = float(np.mean(ppo_terms(ratios_ext_a5, advantages_a5, eps=0.2))) # PPO objective for extreme update.
raw_mod_a5 = float(np.mean(ratios_mod_a5 * advantages_a5)) # Unclipped moderate objective.
raw_ext_a5 = float(np.mean(ratios_ext_a5 * advantages_a5)) # Unclipped extreme objective.
print("raw objectives:", round(raw_mod_a5, 3), round(raw_ext_a5, 3)) # Inspect deceptive raw gain.
print("PPO objectives:", round(ppo_mod_a5, 3), round(ppo_ext_a5, 3)) # Inspect clipped gain.

In [ ]:
kl_mod_a5 = kl_div(old_pi_a5, moderate_pi_a5) # KL for moderate update.
kl_ext_a5 = kl_div(old_pi_a5, extreme_pi_a5) # KL for extreme update.
print("KL moderate:", round(kl_mod_a5, 3), "KL extreme:", round(kl_ext_a5, 3)) # Inspect policy movement.
assert kl_ext_a5 > 10 * kl_mod_a5 # Verify the extreme update is much farther away.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a safety diagnostic plot.
labels_a5 = ["moderate", "extreme"] # Label the two candidate updates.
plt.bar(np.arange(2) - 0.18, [raw_mod_a5, raw_ext_a5], width=0.36, label="unclipped", color="gray") # Plot raw surrogate.
plt.bar(np.arange(2) + 0.18, [ppo_mod_a5, ppo_ext_a5], width=0.36, label="PPO clipped", color="seagreen") # Plot clipped surrogate.
plt.xticks(np.arange(2), labels_a5) # Label candidate policies.
plt.title("Advanced 5: clipping deflates an extreme jump") # Title the plot.
plt.ylabel("mean surrogate") # Label objective axis.
plt.legend() # Show labels.
plt.show() # Display the diagnostic.

▶ What you'll see: the raw surrogate loves the extreme jump, but PPO clipping removes much of that extra credit and KL exposes the huge policy movement.

👀 Takeaway: TRPO and PPO are safety devices for policy optimization: they improve probabilities while resisting jumps the old batch cannot justify.